# Evaluation — llama3.3:70b

Compares `llama3.3-70b` LLM predictions against manually labeled ground truth (`data/manuallabeled/*_dmp.json`).

Sections:
1. Load samples & accuracy table
2. Per-sample accuracy chart
3. Confusion matrix
4. Precision / Recall / F1 per label
5. Mislabeled blocks

## Setup

In [1]:
MODEL_NAME  = "llama3.3:70b"
MODEL_BASE  = "llama3.3-70b"
MODEL_TAG   = MODEL_BASE
MODEL_COLOR = "#2563eb"

In [2]:
import sys
sys.path.insert(0, "..")

from evaluate import extract_gold, evaluate_sample, match, LABELS, SHORT, LLM_DIR, MANUAL_DIR, NO_MATCH
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# Publication style
sns.set_theme(style="ticks", palette="muted")
plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white",
    "axes.edgecolor":"#555555","axes.linewidth":0.8,
    "grid.color":"#dddddd","grid.linewidth":0.5,
    "text.color":"#111111","axes.labelcolor":"#111111",
    "xtick.color":"#111111","ytick.color":"#111111",
    "font.size":11,"axes.titlesize":13,"axes.labelsize":11,
    "xtick.labelsize":10,"ytick.labelsize":10,
    "legend.fontsize":10,"legend.frameon":True,"legend.edgecolor":"#cccccc",
    "figure.dpi":120,
})

def _snum(name):
    s = name.stem if hasattr(name,"stem") else name
    return int(s.replace("_dmp","").replace("sample",""))

print(f"Ready — model: {MODEL_NAME}  tag: {MODEL_TAG}")


## 1 — Load samples & accuracy table

In [ ]:
samples = sorted(MANUAL_DIR.glob("*_dmp.json"), key=_snum)
conf_all = defaultdict(lambda: defaultdict(int))
per_sample_rows, all_errors = [], []

for mp in samples:
    stem = mp.stem.replace("_dmp", "")
    pp   = LLM_DIR / f"{stem}_{MODEL_TAG}.json"
    if not pp.exists():
        print(f"  MISSING: {pp.name}"); continue
    gold = extract_gold(mp)
    conf = evaluate_sample(pp, gold)
    blocks = json.loads(pp.read_text(encoding="utf-8"))
    for block in blocks:
        tl = match(block["text"], gold)
        pl = block.get("label", "answer.text")
        key = tl if tl else NO_MATCH
        conf_all[key][pl] += 1
        if tl != pl:
            all_errors.append({"sample": stem, "text": block["text"][:120],
                                "true": tl or "no_match", "pred": pl,
                                "page": block.get("page", "-")})
    tp = sum(conf.get(l, {}).get(l, 0) for l in LABELS)
    n  = sum(sum(v.values()) for v in conf.values())
    per_sample_rows.append({"sample": stem, "total": n, "correct": tp,
                             "errors": n - tp, "accuracy": tp / n if n else 0,
                             "formula": f"{tp}/{n}"})

df     = pd.DataFrame(per_sample_rows)
df_err = pd.DataFrame(all_errors)

hdr = f"{'Sample':<12}  {'Total':>5}  {'Correct':>7}  {'Errors':>6}  {'Accuracy':>8}  Formula"
print(hdr); print("-" * len(hdr))
for row in df.itertuples():
    print(f"{row.sample:<12}  {row.total:>5}  {row.correct:>7}  {row.errors:>6}  {row.accuracy*100:>7.1f}%  {row.formula}")
print("-" * len(hdr))
tn, tc = df["total"].sum(), df["correct"].sum()
print(f"{'TOTAL':<12}  {tn:>5}  {tc:>7}  {tn-tc:>6}  {tc/tn*100:>7.1f}%  {tc}/{tn}")


## 2 — Per-sample accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2563eb" if a >= 0.95 else "#f59e0b" if a >= 0.85 else "#dc2626"
          for a in df["accuracy"]]
bars = ax.bar(df["sample"], df["accuracy"] * 100,
              color=colors, width=0.6, edgecolor="white")
for bar, acc in zip(bars, df["accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc*100:.1f}%", ha="center", va="bottom", fontsize=9)
avg = df["accuracy"].mean() * 100
ax.axhline(avg, color="#444444", linestyle="--", linewidth=1.0)
ax.text(len(df) - 0.45, avg + 0.7, f"Mean = {avg:.1f}%",
        ha="right", va="bottom", fontsize=9.5, color="#444444")
ax.set_ylim(60, 112)
ax.set_ylabel("Block-level accuracy (%)")
ax.set_xlabel("Sample")
ax.set_title(f"Per-sample label accuracy — {MODEL_NAME}\n(unmatched blocks counted as errors)", pad=10)
ax.tick_params(axis="x", rotation=30)
legend_h = [mpatches.Patch(facecolor="#2563eb", label="≥ 95%"),
            mpatches.Patch(facecolor="#f59e0b", label="85–94%"),
            mpatches.Patch(facecolor="#dc2626", label="< 85%")]
ax.legend(handles=legend_h, title="Accuracy band", loc="lower left", framealpha=0.9)
sns.despine(); plt.tight_layout(); plt.show()


## 3 — Confusion matrix (all samples combined)

In [ ]:
matrix = pd.DataFrame(
    [[conf_all.get(t,{}).get(p,0) for p in LABELS] for t in LABELS],
    index=SHORT, columns=SHORT)
matrix_norm = matrix.div(matrix.sum(axis=1).replace(0,1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", linewidths=0.6,
            linecolor="white", ax=axes[0], cbar=False, annot_kws={"size":13})
axes[0].set_title("Raw counts", pad=10, fontsize=12)
axes[0].set_xlabel("Predicted label", labelpad=8)
axes[0].set_ylabel("True label", labelpad=8)
sns.heatmap(matrix_norm, annot=True, fmt=".0%", cmap="Blues", linewidths=0.6,
            linecolor="white", ax=axes[1], cbar=False, vmin=0, vmax=1,
            annot_kws={"size":12})
axes[1].set_title("Row-normalised (recall per label)", pad=10, fontsize=12)
axes[1].set_xlabel("Predicted label", labelpad=8)
axes[1].set_ylabel("True label", labelpad=8)
plt.suptitle(f"Confusion matrix — {MODEL_NAME} — all samples", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


## 4 — Precision / Recall / F1 per label

In [ ]:
f1_rows = []
for lbl in LABELS:
    tp = conf_all.get(lbl,{}).get(lbl,0)
    fp = sum(conf_all.get(o,{}).get(lbl,0) for o in LABELS if o!=lbl)
    fn = sum(v for p,v in conf_all.get(lbl,{}).items() if p!=lbl)
    support = tp+fn
    p_  = tp/(tp+fp) if (tp+fp) else 0.
    r_  = tp/support if support else 0.
    f1  = 2*p_*r_/(p_+r_) if (p_+r_) else 0.
    f1_rows.append({"label":lbl,"precision":p_,"recall":r_,"f1":f1,"support":support})
df_f1 = pd.DataFrame(f1_rows)

x, w = range(len(LABELS)), 0.26
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar([i-w for i in x], df_f1["precision"]*100, width=w, label="Precision",
            color="#2563eb", edgecolor="white")
b2 = ax.bar([i   for i in x], df_f1["recall"]*100,    width=w, label="Recall",
            color="#16a34a", edgecolor="white")
b3 = ax.bar([i+w for i in x], df_f1["f1"]*100,        width=w, label="F1",
            color="#dc2626", edgecolor="white")
for bar, row in zip(b3, df_f1.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f"{row.f1*100:.0f}%", ha="center", va="bottom",
            fontsize=9, fontweight="bold", color="#dc2626")
ax.set_xticks(list(x)); ax.set_xticklabels(SHORT, fontsize=10)
ax.set_ylim(0, 118); ax.set_ylabel("Score (%)")
ax.set_title(f"Precision, Recall, F1 per label — {MODEL_NAME}", pad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v:.0f}%"))
ax.legend(loc="upper right", framealpha=0.9)
sns.despine(); plt.tight_layout(); plt.show()

df_f1.set_index("label").style.format(
    {"precision":"{:.1%}","recall":"{:.1%}","f1":"{:.1%}","support":"{:.0f}"})


## 5 — Mislabeled blocks

In [ ]:
if df_err.empty:
    print("No errors — perfect score!")
else:
    print(f"{len(df_err)} mislabeled blocks\n")
    breakdown = (df_err.groupby(["true","pred"]).size()
                 .reset_index(name="count")
                 .sort_values("count", ascending=False))
    print(breakdown.to_string(index=False))


In [ ]:
if not df_err.empty:
    pd.set_option("display.max_colwidth", 100)
    display(df_err[["sample","page","true","pred","text"]])
